# Encoder-Only Transformer model

**Auteur**: `Jakob De Vreese`  
**Bachelorproef**: Overdraagbare unsupervised anomaliedetectie bij hybride HVAC-systemen  
**Academiejaar**: `2025-2026`  

## Doel

Het implementeren van een **Encoder-Only Transformer** architectuur voor de detectie van anomalieën in _hybride HVAC-systemen_, gebaseerd op de paper `Transformer encoder based self-supervised learning for HVAC fault detection with unlabeled data` [(Abdollah et al., 2024)](http://dx.doi.org/10.1016/j.buildenv.2024.111568}).

### Verwachte brondata

Pre-processed multivariate dataset met HVAC-systeemdata (bijv. 10-minuten resolutie).

### Concept lagen

#### A. Pre-processing & Embedding

- **Normalisatie**
- **Time2Vec**
- **Learnable Positional Encoding**

#### B. Self-Supervised Learning (Pre-training)

- **Markov Chain Masking**
- **Masked MSE Loss**

#### C. Anomaliedetectie en Diagnostiek

- **Anomaliescore**
- **Peak Over Threshold (POT)**

### Stappenplan

- **Stap 1**: Data processing (split train, val, test e.a.)
- **Stap 2**: De Encoder-Only Architectuur (Keras/TensorFlow implementatie inclusief Time2Vec)
- **Stap 3**: Self-Supervised Training (Markov chain block masking & MSE-loss)
- **Stap 4**: Keras Training via overschreven `train_step`
- **Stap 5**: Hyperparameter Tuning (KerasTuner) & Model Opslaan
- **Stap 6**: Inferentie & Anomaliescores
- **Stap 7**: Validatie met Synthetische data

In [20]:
import pandas as pd
import numpy as np
import keras
from sklearn.preprocessing import StandardScaler
import joblib
import tensorflow as tf
from tensorflow.keras import layers

## STAP 1 - Data Preprocessing

Voordat de data in het Keras-model wordt gevoed, wordt de volgende voorbereiding uitgevoerd:

1. **Data inlezen**
2. **Feature reductie**
3. **Data opsplitsen**
4. **Normalisatie**
5. **Windowing (3d-tensors)**

### 1.1 Dataset inlezen

In [13]:
# Dataset inlezen
GEBOUW = 'dunant1'

url = f'../02_eda_en_ground_truth/processed/{GEBOUW}_train.csv'
data = pd.read_csv(url)
data.head()

,timestamp,f_1,f_2,f_3,f_4,f_6,f_7,f_8,f_9,f_10,...,f_50,f_51,f_52,f_53,f_54,f_55,is_weekdag,is_weekend,is_werkuur,uur
0,2026-03-09 00:10:00+00:00,7.0,0.77,1.0,39.877285,31.832268,1.0,23.594538,44.640636,1.0,...,0.0,3.851797,99.589400,0.749325,86.946641,0.0,1,0,0,0
1,2026-03-09 00:20:00+00:00,9.0,0.78,1.0,41.921010,34.104786,1.0,46.930084,44.640636,1.0,...,0.0,3.903977,100.000000,0.559565,45.771835,0.0,1,0,0,0
2,2026-03-09 00:30:00+00:00,8.0,0.86,1.0,43.412975,35.585346,1.0,43.235270,44.640636,0.0,...,0.0,4.244891,99.196013,0.322963,91.228401,0.0,1,0,0,0
3,2026-03-09 00:40:00+00:00,7.0,0.78,1.0,43.734930,35.924030,1.0,48.845810,44.640636,0.0,...,0.0,4.448537,98.800579,0.300000,323.796930,0.0,1,0,0,0
4,2026-03-09 00:50:00+00:00,9.0,0.78,1.0,44.037724,36.229050,1.0,44.068630,44.640636,0.0,...,0.0,4.748848,99.259927,0.345344,204.933704,0.0,1,0,0,0


In [14]:
print(data.shape)

(4146, 59)


### 1.2 Feature reductie

Om het model zo efficiënt mogelijk te maken verwijderen ze in de paper één feature per paar met een correlatie **boven 95%**.

In [15]:
# Bereken correlatie
corr_matrix = data.drop(columns=['timestamp']).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Identificeer kolommen die WEG moeten
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

# Identificeer kolommen die we BEHOUDEN (behalve timestamp)
kept_features = [col for col in data.columns if col not in to_drop and col != 'timestamp']

print(f"Aantal behouden features: {len(kept_features)}")
# Sla deze lijst op voor later!
import json
with open(f'encoder_only/features_{GEBOUW}.json', 'w') as f:
    json.dump(kept_features, f)

data_filtered = data[kept_features]

Aantal behouden features: 44


### 1.3 Data opspliten

In [16]:
n = len(data)
train_df = data_filtered[0:int(n*0.7)]
val_df = data_filtered[int(n*0.7):int(n*0.85)]
test_df = data_filtered[int(n*0.85):]

### 1.4 Normalisatie (Z-score Scaling)

Volgens de paper moeten we de data standaardiseren. We normaliseren `train_df` en gebruiken de parameters om dit ook op de rest toe te passen.

In [18]:
scaler = StandardScaler()

# Fit alleen op training data
scaler.fit(train_df)

# Transformeer alle sets
train_scaled = scaler.transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

# Sla de scaler op voor later gebruik in productie!
joblib.dump(scaler, f'encoder_only/scaler_{GEBOUW}.pkl')

['encoder_only/scaler_dunant1.pkl']

### 1.5 Windowing (3D Tensor maken)

De `Transformer` heeft een venster nodig van data om patronen te herkennen. In de paper gebruikt men venstergroottes ($w$) van 1 dag of wel 144 tijdstappen.

In [19]:
def create_windows(data_array, window_size=144):
    X = []
    for i in range(len(data_array) - window_size):
        X.append(data_array[i:i + window_size])
    return np.array(X)

# Maak de 3D tensors: (samples, window_size, features)
X_train = create_windows(train_scaled, window_size=144)
X_val = create_windows(val_scaled, window_size=144)
X_test = create_windows(test_scaled, window_size=144)

print(f"Shape training data: {X_train.shape}")

Shape training data: (2758, 144, 44)


## STAP 2 - De Encoder-Only Architectuur

In deze stap bouwen we het model in `Keras`. We wijken af van de standaard encoder-decoder structuur en gebruiken een **encoder-only** model, wat de comutationele complexiteit verlaagt.

### De lagen

#### Preprocessing & encoding

- **Normalization**
- **Linear projection**
- **Time2Vec**
- **Learnable Positional Encoding**

#### Transformer encoder blocs

- **Multi-Head Attention (MHA)**
- **Point-wise Feedforward Netword (FFN)**
- **Residual Connections & LayerNorm**

#### Reconstruction Step

- **Output Projection**



### Time2Vec

In [21]:
class Time2Vec(layers.Layer):
    def __init__(self, kernel_size=1):
        super(Time2Vec, self).__init__()
        self.k = kernel_size

    def build(self, input_shape):
        # Gewichten voor de lineaire component (trend)
        self.wb = self.add_weight(name='wb', shape=(input_shape[1], 1), initializer='uniform', trainable=True)
        self.bb = self.add_weight(name='bb', shape=(input_shape[1], 1), initializer='uniform', trainable=True)
        # Gewichten voor de periodieke component (sinus)
        self.wa = self.add_weight(name='wa', shape=(input_shape[1], self.k), initializer='uniform', trainable=True)
        self.ba = self.add_weight(name='ba', shape=(input_shape[1], self.k), initializer='uniform', trainable=True)

    def call(self, inputs):
        # Inputs zijn hier de tijdstappen (bijv. van 0 tot 143)
        bias = self.wb * inputs + self.bb
        # Periodieke activatie om cycli te herkennen
        dp = tf.sin(self.wa * inputs + self.ba)
        return tf.concat([bias, dp], axis=-1)

### Transformer Encoder blok

In [22]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # 1. Multi-Head Attention[cite: 1]
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    # 2. Add & Norm (Residual Connection)[cite: 1]
    res = x + inputs
    x = layers.LayerNormalization(epsilon=1e-6)(res)

    # 3. Point-wise Feedforward Network[cite: 1]
    x = layers.Dense(ff_dim, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(inputs.shape[-1])(x)
    # 4. Add & Norm[cite: 1]
    x = x + res
    return layers.LayerNormalization(epsilon=1e-6)(x)

### Het volledige model

In [23]:
def build_model(window_size, num_features, d_model, num_heads, ff_dim, num_layers, dropout):
    inputs = layers.Input(shape=(window_size, num_features))
    
    # --- STAP 1: Preprocessing & Encoding ---
    # Normalisatie voor trainingsstabiliteit[cite: 1]
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    
    # Lineaire projectie naar model dimensie 'd'[cite: 1]
    x = layers.Dense(d_model)(x)
    
    # Learnable Positional Encoding toevoegen[cite: 1]
    pos_emb = layers.Embedding(input_dim=window_size, output_dim=d_model)(tf.range(start=0, limit=window_size, delta=1))
    x = x + pos_emb

    # --- STAP 2: Transformer Processing ---
    # Stapel 'B' aantal encoder layers[cite: 1]
    for _ in range(num_layers):
        x = transformer_encoder(x, d_model, num_heads, ff_dim, dropout)

    # --- STAP 3: Reconstruction Step ---
    # Terugmappen naar de originele dimensie 'm'[cite: 1]
    outputs = layers.Dense(num_features)(x)
    
    return tf.keras.Model(inputs, outputs)

In [24]:
# Voorbeeld initialisatie met parameters uit de paper
model = build_model(
    window_size=144, 
    num_features=len(kept_features), 
    d_model=64, 
    num_heads=7, 
    ff_dim=64, 
    num_layers=2, 
    dropout=0.1
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 144, 44)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 144, 44)   │         88 │ input_layer[0][0] │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 144, 64)   │      2,880 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 144, 64)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 144, 64)   │    116,096 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 144, 64)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 144, 64)   │          0 │ dropout_1[0][0],  │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 144, 64)   │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 144, 64)   │      4,160 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 144, 64)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 144, 64)   │      4,160 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 144, 64)   │          0 │ dense_2[0][0],    │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 144, 64)   │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 144, 64)   │    116,096 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 144, 64)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 144, 64)   │          0 │ dropout_4[0][0],  │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 144, 64)   │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 144, 64)   │      4,160 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 144, 64)   │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 144, 64)   │      4,160 │ dropout_5[0][0] 

 Total params: 255,172 (996.77 KB)

 Trainable params: 255,172 (996.77 KB)

 Non-trainable params: 0 (0.00 B)

## Stap 3 - Self-Supervised Pre-training (Markov Chain Masking)

We gebruiken een Markov-keten met twee toestanden (Masked/Unmasked). De lengte van de segmenten volgt een geometrische distributie met 2 parameters:

- $l_m$: De gemiddelde lengte van een verborgen segment (de paper stelt $l_m=3$ voor)
- $r$: De proportie van de totale data die we willen maskeren (vaak rond 15-20%)

De gemiddelde lengte van een ongemaskeerd element word dan berekend als

$$
l_u = \frac{1 - r}{r} . l_m
$$

In [ ]:
def generate_markov_mask(input_shape, r=0.15, lm=3):
    """
    Genereert een binaire mask matrix (M) op basis van een Markov Chain.
    Input_shape: (window_size, num_features)
    """
    W, M = input_shape
    # Bereken de gemiddelde ongemaskeerde lengte
    lu = ((1 - r) / r) * lm
    
    # Transitie kansen (p_m: 1->0, p_u: 0->1)
    p_m = 1 / lu
    p_u = 1 / lm
    
    mask = np.ones(input_shape)
    
    for j in range(M): # Per feature
        curr_state = 1 # Start als unmasked
        for i in range(W):
            if curr_state == 1:
                if np.random.rand() < p_m:
                    curr_state = 0
            else:
                if np.random.rand() < p_u:
                    curr_state = 1
            mask[i, j] = curr_state
            
    return mask

Dit is een cruciale stap: we willen dat het model alleen gestraft wordt voor fouten in de segmenten die we verborgen hebben. We berekenen de verlieswaarde dus uitsluitend op de posities waar de mask waarde 0 is.

In [26]:
def masked_mse_loss(y_true, y_pred, mask):
    """
    Berekent MSE uitsluitend op de gemaskeerde segmenten.
    """
    # De paper berekent loss enkel op de verborgen delen (waar mask == 0)
    # We maken een invers masker: 1 voor verborgen, 0 voor zichtbaar
    inverse_mask = 1.0 - tf.cast(mask, tf.float32)
    
    sq_diff = tf.square(y_true - y_pred) * inverse_mask
    
    # Gemiddelde nemen over het aantal gemaskeerde punten
    loss = tf.reduce_sum(sq_diff) / (tf.reduce_sum(inverse_mask) + 1e-6)
    return loss

We maken een klasse die het basismodel (de Transformer) inkapselt en de train_step overschrijft om de Markov-maskering toe te passen.

In [32]:
class HVACModel(tf.keras.Model):
    def __init__(self, transformer_model, r=0.15, lm=3):
        super(HVACModel, self).__init__()
        self.transformer = transformer_model
        self.r = r
        self.lm = lm
        # Metric om de voortgang van de reconstructie te volgen
        self.loss_tracker = tf.keras.metrics.Mean(name="masked_mse")

    def train_step(self, data):
        # Data is hier een batch van (batch_size, window_size, features)
        x = data
        batch_size = tf.shape(x)[0]
        window_size = tf.shape(x)[1]
        num_features = tf.shape(x)[2]

        # 1. Genereer het Markov Masker (on-the-fly tijdens training)
        # We doen dit in Python/Numpy voor de Markov-logica en converteren naar Tensor
        mask = tf.numpy_function(
            func=generate_markov_mask,
            inp=[(window_size, num_features), self.r, self.lm],
            Tout=tf.float64
        )
        mask = tf.cast(mask, dtype=tf.float32)
        mask.set_shape((window_size, num_features))
        mask = tf.repeat(tf.expand_dims(mask, 0), batch_size, axis=0)

        # 2. Pas het masker toe: verberg segmenten door ze 0 te maken
        x_masked = x * mask

        with tf.GradientTape() as tape:
            # 3. Forward pass: het model probeert de hele reeks te reconstrueren
            y_pred = self.transformer(x_masked, training=True)
            
            # 4. Bereken de loss ENKEL op de gemaskeerde delen (mask == 0)
            # De paper stelt dat we alleen gestraft worden voor wat we niet konden zien
            loss = masked_mse_loss(x, y_pred, mask)

        # 5. Gradiënten berekenen en toepassen
        trainable_vars = self.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        # Update tracker
        self.loss_tracker.update_state(loss)
        return {"masked_mse": self.loss_tracker.result()}

    @property
    def metrics(self):
        # Zorgt ervoor dat de tracker gereset wordt na elke epoch
        return [self.loss_tracker]

    def call(self, inputs):
        # Voor inferentie gebruiken we gewoon de transformer
        return self.transformer(inputs)

In [33]:
# Maak de kern
base_transformer = build_model(
    window_size=144, 
    num_features=len(kept_features),
    d_model=64, 
    num_heads=7, 
    ff_dim=128, 
    num_layers=2, 
    dropout=0.1
)

# Maak de HVAC wrapper voor self-supervised training
hvac_model = HVACModel(base_transformer, r=0.15, lm=3)
hvac_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))

## Stap 4: Keras Training & Monitoring

In deze fase brengen we alles samen. Hoewel we de train_step handmatig hebben overschreven, gebruiken we de krachtige standaardfunctionaliteiten van Keras voor de uitvoering.

### Kernonderdelen van de training

- Optimizer (Adam): De paper maakt gebruik van de Adam optimizer, die de leersnelheid tijdens het proces dynamisch aanpast voor snellere convergentie.  
- EarlyStopping: Dit stopt de training automatisch zodra de loss op de validatieset niet meer verbetert, wat essentieel is bij Transformers om onnodige rekenkracht te besparen.  
- ModelCheckpoint: Omdat de loss tijdens de training kan fluctueren door de dynamische maskering, slaan we alleen de versie van het model op die het beste presteert op de validatiedata.  
- Validation: We monitoren de prestaties op data die het model niet gebruikt voor de gewichtsaanpassingen, om te verifiëren of de geleerde correlaties overdraagbaar zijn naar nieuwe situaties.

### 4.1 Callbacks en Optimizer configureren

Voordat we de training starten, definiëren we de "bewakers" van ons proces. Deze zorgen ervoor dat we niet blindelings blijven trainen.

In [34]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# 1. Optimizer: Adam met een bescheiden leersnelheid (LR) zoals in de paper
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# 2. EarlyStopping: Stop als de validatie-loss 10 epochs niet meer zakt
early_stopping = EarlyStopping(
    monitor='val_masked_mse', 
    patience=10, 
    restore_best_weights=True
)

# 3. ModelCheckpoint: Sla de beste gewichten op in een bestand
checkpoint = ModelCheckpoint(
    filepath=f'encoder_only/best_model_{GEBOUW}.weights.h5',
    monitor='val_masked_mse',
    save_best_only=True,
    save_weights_only=True # We slaan gewichten op omdat we een custom model gebruiken
)

# 4. ReduceLROnPlateau: Verlaag de LR als de training stagneert
reduce_lr = ReduceLROnPlateau(
    monitor='val_masked_mse', 
    factor=0.5, 
    patience=5, 
    min_lr=1e-6
)

callbacks = [early_stopping, checkpoint, reduce_lr]

### 4.2 De Training uitvoeren

We roepen nu de .fit() methode aan. Let op: we geven X_train mee als zowel de input als de target, maar onze custom train_step zal intern de maskering regelen en de loss berekenen.

In [35]:
# Compileer het HVAC wrapper model
hvac_model.compile(optimizer=optimizer)

# Start het leerproces
history = hvac_model.fit(
    X_train, 
    epochs=100,            # Maximaal 100, EarlyStopping grijpt waarschijnlijk eerder in
    batch_size=32,         # Batch grootte aanpasbaar voor GPU/CPU geheugen
    validation_data=(X_val, X_val), 
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100


TypeError: Dimension value must be integer or None or have an __index__ method, got value '<tf.Tensor 'strided_slice_1:0' shape=() dtype=int32>' with type '<class 'tensorflow.python.framework.ops.SymbolicTensor'>'

In [ ]:
# Hyperparameters
NUM_EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

In [ ]:
# We casten de data expliciet naar float32 voor compatibiliteit met Keras/TensorFlow
X_train_tf = tf.cast(X_train, tf.float32)
X_val_tf = tf.cast(X_val, tf.float32)

# TensorFlow Datasets aanmaken voor efficiënte batching en shuffling
train_dataset = tf.data.Dataset.from_tensor_slices(X_train_tf)
train_dataset = train_dataset.shuffle(buffer_size=1024, seed=SEED, reshuffle_each_iteration=False)
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices(X_val_tf)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Optimizer initialiseren
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

### 4.3 De Iteratieve Epoch Loop

Een *epoch* is voltooid wanneer het model de volledige dataset (`train_dataset`) exact één keer heeft doorlopen in batches. We itereren over het gedefinieerde aantal epochs en houden de gemiddelde foutmarge (loss) per epoch bij. Als de training correct verloopt, zou deze gemiddelde loss na elke epoch stapsgewijs moeten dalen totdat het model convergeert.

In [ ]:
print("Start van de training...")

# Optimizer koppelen aan het model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE))

# Callback voor automatische stop als de validatie loss niet meer verbetert
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

# Trainen met de ingebouwde fit-methode
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=NUM_EPOCHS,
    callbacks=[early_stopping]
)

print("Training voltooid.")

In [ ]:
def predict_with_attention(model, X, batch_size=32):
    recon_batches = []
    attn_batches = []
    for i in range(0, len(X), batch_size):
        batch = tf.cast(X[i : i + batch_size], tf.float32)
        recon, attn = model(batch, training=False, return_attention=True)
        recon_batches.append(recon.numpy())
        attn_batches.append(attn.numpy())
    return np.concatenate(recon_batches, axis=0), np.concatenate(attn_batches, axis=0)

def compute_attention_weighted_scores(X, recon, attn_scores, feature_reduce="max"):
    # Gemiddelde attention per tijdstap (heads + query)
    attn_mean = attn_scores.mean(axis=1)  # (batch, w, w)
    attn_t = attn_mean.mean(axis=1)       # (batch, w)
    errors = np.abs(recon - X)
    weighted = errors * attn_t[..., None]
    window_scores = weighted.mean(axis=(1, 2))
    if feature_reduce == "max":
        feature_scores = weighted.max(axis=1)
    else:
        feature_scores = weighted.mean(axis=1)
    return window_scores, feature_scores, attn_t

def fit_pot_thresholds(feature_scores, q=0.01, initial_percentile=85, min_excess=10):
    thresholds = np.zeros(feature_scores.shape[1], dtype=np.float32)
    for f in range(feature_scores.shape[1]):
        scores = feature_scores[:, f]
        t = np.percentile(scores, initial_percentile)
        excesses = scores[scores > t] - t
        if len(excesses) < min_excess:
            thresholds[f] = t
            continue
        c, _, scale = genpareto.fit(excesses, floc=0)
        thresholds[f] = t + genpareto.ppf(1 - q, c, loc=0, scale=scale)
    return thresholds

def predict_window_anomalies(feature_scores, thresholds):
    return (feature_scores > thresholds[None, :]).any(axis=1).astype(int)

In [ ]:
def compute_weighted_errors(X, recon, attn_scores):
    attn_mean = attn_scores.mean(axis=1)  # (batch, w, w)
    attn_t = attn_mean.mean(axis=1)       # (batch, w)
    errors = np.abs(recon - X)
    weighted = errors * attn_t[..., None]
    return weighted, attn_t

def aggregate_to_timestep_series(weighted_errors, series_length, window_size, stride):
    sums = np.zeros((series_length, weighted_errors.shape[-1]), dtype=np.float32)
    counts = np.zeros((series_length, 1), dtype=np.float32)
    for i in range(weighted_errors.shape[0]):
        start = i * stride
        end = start + window_size
        if end > series_length:
            break
        sums[start:end] += weighted_errors[i]
        counts[start:end] += 1.0
    return sums / np.maximum(counts, 1.0)

def rolling_pot_thresholds(series_scores, q=0.01, initial_percentile=85, window_len=12, min_excess=10):
    thresholds = np.full_like(series_scores, np.nan, dtype=np.float32)
    for t in range(window_len, series_scores.shape[0] + 1):
        window = series_scores[t - window_len : t]
        for f in range(series_scores.shape[1]):
            scores = window[:, f]
            t0 = np.percentile(scores, initial_percentile)
            excesses = scores[scores > t0] - t0
            if len(excesses) < min_excess:
                thresholds[t - 1, f] = t0
                continue
            c, _, scale = genpareto.fit(excesses, floc=0)
            thresholds[t - 1, f] = t0 + genpareto.ppf(1 - q, c, loc=0, scale=scale)
    return thresholds

def window_labels_from_timestep(timestep_flags, window_size, stride):
    labels = []
    for start in range(0, len(timestep_flags) - window_size + 1, stride):
        labels.append(1 if np.any(timestep_flags[start : start + window_size]) else 0)
    return np.array(labels, dtype=int)

### 4.4 Evaluatie van het initieel getrainde model

Om de prestaties van het initieel getrainde model objectief te beoordelen, laden we opnieuw de gelabelde ground truth testset in. De timestep-labels worden omgezet naar window-level labels, zodat elk sliding window een binair label krijgt: normaal of anomalie. Daarna laten we het getrainde model reconstructies maken op deze gelabelde data, berekenen we de anomaliescore per window en vergelijken we die scores met een drempel die enkel uit de trainingsdata is afgeleid. Zo vermijden we dat de evaluatie zelf informatie uit de testlabels lekt. Deze baseline vormt later een directe vergelijking met het best getunede model uit Stap 5.

In [ ]:
# 4.4.1 Ground truth en labels inladen
synth_csv = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test.csv'
labels_npy = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test_labels.npy'

synth_df = pd.read_csv(synth_csv)
y_true_timestep = np.load(labels_npy).astype(int)

# Verwijder mogelijke index- of timestampkolommen
if 'timestamp' in synth_df.columns:
    synth_df = synth_df.drop(columns=['timestamp'])

for col in list(synth_df.columns):
    if col.lower().startswith('unnamed'):
        synth_df = synth_df.drop(columns=[col])

# Zorg dat de kolommen exact overeenkomen met de trainingsfeatures
synth_df = synth_df[train_data.columns]

# Zelfde normalisatie als tijdens training
synth_norm = scaler.transform(synth_df)

In [ ]:
# Maak sliding windows van de ground truth set
STRIDE = 4 # halve dag overlap

# Sliding windows bouwen
X_eval = create_windows(synth_norm, WINDOW_SIZE, step=STRIDE)

In [ ]:
# Timestep-labels omzetten naar window-labels:
# een window is anomalie als minstens één timestep anomalie is
y_true_window = np.array(
    [1 if np.max(y_true_timestep[i:i + WINDOW_SIZE]) > 0 else 0
     for i in range(0, len(y_true_timestep) - WINDOW_SIZE + 1, STRIDE)],
    dtype=int
)

In [ ]:
print(f"Eval rows: {synth_df.shape[0]}")
print(f"Eval windows: {X_eval.shape}")
print(f"Window labels: {y_true_window.shape}, anomalies={y_true_window.sum()}")

In [ ]:
# Scores op de validatieset gebruiken om de thresholds te kalibreren
recon_val, attn_val = predict_with_attention(model, X_val, batch_size=BATCH_SIZE)
val_window_scores, val_feature_scores, _ = compute_attention_weighted_scores(X_val, recon_val, attn_val)

feature_thresholds = fit_pot_thresholds(
    val_feature_scores, q=POT_Q, initial_percentile=POT_PERCENTILE
 )
window_threshold_plot = np.percentile(val_window_scores, 99)

In [ ]:
# Scores op de evaluatieset (gelabelde synthetische windows)
recon_eval, attn_eval = predict_with_attention(model, X_eval, batch_size=BATCH_SIZE)
eval_window_scores, eval_feature_scores, _ = compute_attention_weighted_scores(X_eval, recon_eval, attn_eval)

In [ ]:
# Rolling POT (2u) per feature op timestep-niveau
weighted_eval, _ = compute_weighted_errors(X_eval, recon_eval, attn_eval)
timestep_scores_eval = aggregate_to_timestep_series(
    weighted_eval, len(synth_norm), WINDOW_SIZE, STRIDE
 )
rolling_thresholds_eval = rolling_pot_thresholds(
    timestep_scores_eval, q=POT_Q, initial_percentile=POT_PERCENTILE, window_len=ROLLING_WINDOW_STEPS
 )
timestep_anom_eval = (timestep_scores_eval > rolling_thresholds_eval).any(axis=1)
y_pred_eval_roll = window_labels_from_timestep(timestep_anom_eval, WINDOW_SIZE, STRIDE)

# Gebruik rolling POT voor evaluatie
y_pred_eval = y_pred_eval_roll

In [ ]:
# Optioneel: statische POT voor vergelijking
y_pred_eval_static = predict_window_anomalies(eval_feature_scores, feature_thresholds)

In [ ]:
print("feature threshold percentiles:", np.percentile(feature_thresholds, [50, 90, 95, 99]))

In [ ]:
print("val window:", np.percentile(val_window_scores, [50, 90, 95, 99]))
print("eval window:", np.percentile(eval_window_scores, [1, 5, 50, 95, 99]))
print("plot threshold (p99 val):", window_threshold_plot)
print("eval below plot threshold:", np.mean(eval_window_scores <= window_threshold_plot))
print("shapes:", "y_true_window =", y_true_window.shape, "| y_pred_eval =", y_pred_eval.shape)

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_window, y_pred_eval, average='binary', zero_division=0
)

cm = confusion_matrix(y_true_window, y_pred_eval)
tn, fp, fn, tp = cm.ravel()

roc_auc = roc_auc_score(y_true_window, eval_window_scores)
pr_auc = average_precision_score(y_true_window, eval_window_scores)
accuracy = accuracy_score(y_true_window, y_pred_eval)
balanced_acc = balanced_accuracy_score(y_true_window, y_pred_eval)
mcc = matthews_corrcoef(y_true_window, y_pred_eval)

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1",
        "MCC",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Value": [
        accuracy,
        balanced_acc,
        precision,
        recall,
        f1,
        mcc,
        roc_auc,
        pr_auc
    ]
})

print(metrics_df.to_string(index=False))
print("\nConfusion matrix:")
print(cm)

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
fig.suptitle("Encoder-Only Transformer: Evaluatie van het initieel getrainde model", fontsize=18, fontweight="bold")

# Plot 1: anomaly scores over de windows
ax1 = axes[0, 0]
ax1.plot(eval_window_scores, color="#1f77b4", linewidth=1.2, label="Anomaly score")
ax1.axhline(window_threshold_plot, color="#d62728", linestyle="--", linewidth=2, label=f"Plot threshold = {window_threshold_plot:.4f}")

anom_idx = np.where(y_true_window == 1)[0]
ax1.scatter(
    anom_idx,
    eval_window_scores[anom_idx],
    s=18,
    color="#ff7f0e",
    alpha=0.8,
    label="Ground truth anomalie",
    zorder=5
)

ax1.set_title("Anomaly scores per window")
ax1.set_xlabel("Window index")
ax1.set_ylabel("Score")
ax1.legend(loc="upper right")
ax1.grid(alpha=0.25)

# Plot 2: scoreverdeling per klasse
ax2 = axes[0, 1]
sns.histplot(eval_window_scores[y_true_window == 0], bins=40, kde=True, stat="density", color="#1f77b4", alpha=0.45, label="Normaal", ax=ax2)
sns.histplot(eval_window_scores[y_true_window == 1], bins=40, kde=True, stat="density", color="#d62728", alpha=0.45, label="Anomalie", ax=ax2)
ax2.axvline(window_threshold_plot, color="#2ca02c", linestyle="--", linewidth=2, label="Plot threshold")
ax2.set_title("Scoreverdeling per klasse")
ax2.set_xlabel("Score")
ax2.set_ylabel("Dichtheid")
ax2.legend(loc="upper right")
ax2.grid(alpha=0.25)

# Plot 3: confusion matrix
ax3 = axes[1, 0]
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    square=True,
    xticklabels=["Voorspeld normaal", "Voorspeld anomalie"],
    yticklabels=["Werkelijk normaal", "Werkelijk anomalie"],
    ax=ax3
)
ax3.set_title("Confusion matrix")
ax3.set_xlabel("")
ax3.set_ylabel("")

# Plot 4: metric overview
ax4 = axes[1, 1]
metric_plot_df = metrics_df.copy()
metric_plot_df = metric_plot_df.sort_values("Value", ascending=True)

sns.barplot(
    data=metric_plot_df,
    x="Value",
    y="Metric",
    palette="viridis",
    ax=ax4
)

for i, value in enumerate(metric_plot_df["Value"]):
    ax4.text(min(value + 0.01, 1.02), i, f"{value:.3f}", va="center", fontsize=10)

ax4.set_xlim(0, max(1.0, metric_plot_df["Value"].max() + 0.15))
ax4.set_title("Overzicht van de kernmetrieken")
ax4.set_xlabel("Score")
ax4.set_ylabel("")

plt.show()

## STAP 5 - Hyperparameter Tuning & Model Opslaan

Omdat we een custom architectuur gebruiken, is het belangrijk om de juiste hyperparameters te vinden. Een volledige *Grid Search* is voor Transformers vaak te rekenintensief. Daarom gebruiken we een gecontroleerde iteratieve zoektocht over een beperkt aantal logische combinaties.

Om instabiliteit te vermijden houden we `num_heads` vast op **7** (zoals in de paper) zodat `d_model` steeds deelbaar blijft. We tunen vooral `d_model`, het aantal encoder-lagen, `ffn_expansion`, learning rate en de maskerparameters.

Na de zoektocht slaan we het best presterende model op. Omdat we werken met Keras *Model Subclassing* (custom lagen zoals `Time2Vec`), is de meest robuuste methode het opslaan van de **modelgewichten** (weights) in plaats van het hele modelobject. Dit voorkomt serialisatiefouten bij het inladen.

In [ ]:
def build_model(hp):
    # 1. Definieer de zoekruimte
    d_model = hp.Choice('d_model', values=[112, 140, 168])
    num_heads = 7  # paper setting (d_model blijft deelbaar)
    num_layers = hp.Int('num_layers', min_value=1, max_value=3)
    ffn_expansion = hp.Choice('ffn_expansion', values=[2, 4])
    lr = hp.Choice('learning_rate', values=[2e-4, 1e-4, 5e-5])
    mask_ratio = hp.Choice('mask_ratio', values=[0.15, 0.20])
    block_len = hp.Choice('block_len', values=[3, 4])
    
    # 2. Model instantiëren met de getunede parameters
    temp_model = HVACAnomalyTransformer(
        num_features=NUM_FEATURES,
        d_model=d_model,
        num_heads=num_heads,
        num_layers=num_layers,
        ffn_expansion=ffn_expansion,
        mask_ratio=mask_ratio,
        block_len=block_len,
        mask_seed=SEED
    )
    
    # 3. Dummy aanroep voor initialisatie en compileren
    _ = temp_model(tf.zeros((1, WINDOW_SIZE, NUM_FEATURES)))
    temp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr))
    
    return temp_model

# Configureer de Hyperband Tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_loss',
    max_epochs=12,
    factor=3,
    directory='tuning_logs',
    project_name='hvac_encoder_tuning',
    overwrite=True
 )

print("Start Hyperparameter Tuning met KerasTuner...")

# Zoek naar de beste hyperparameter set
tuner.search(
    train_dataset,
    validation_data=val_dataset,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)]
 )

# Het beste model en de parameters extraheren
best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"\nBeste configuratie gevonden: {best_hps.values}")

### 5.3 Snelle evaluatie beste model

In [ ]:
# Scores op de validatieset gebruiken om de thresholds te kalibreren
recon_val_best, attn_val_best = predict_with_attention(best_model, X_val, batch_size=BATCH_SIZE)
val_window_scores_best, val_feature_scores_best, _ = compute_attention_weighted_scores(
    X_val, recon_val_best, attn_val_best
 )

feature_thresholds_best = fit_pot_thresholds(
    val_feature_scores_best, q=POT_Q, initial_percentile=POT_PERCENTILE
 )
window_threshold_plot_best = np.percentile(val_window_scores_best, 99)

In [ ]:
# Scores op de evaluatieset
recon_eval_best, attn_eval_best = predict_with_attention(best_model, X_eval, batch_size=BATCH_SIZE)
eval_window_scores_best, eval_feature_scores_best, _ = compute_attention_weighted_scores(
    X_eval, recon_eval_best, attn_eval_best
 )

In [ ]:
# Binaire voorspellingen
y_pred_eval_best = predict_window_anomalies(eval_feature_scores_best, feature_thresholds_best)

print("val window:", np.percentile(val_window_scores_best, [50, 90, 95, 99]))
print("eval window:", np.percentile(eval_window_scores_best, [1, 5, 50, 95, 99]))
print("plot threshold (p99 val):", window_threshold_plot_best)
print("eval below plot threshold:", np.mean(eval_window_scores_best <= window_threshold_plot_best))
print("shapes:", "y_true_window =", y_true_window.shape, "| y_pred_eval_best =", y_pred_eval_best.shape)

In [ ]:
# Scorekaart
precision_best, recall_best, f1_best, _ = precision_recall_fscore_support(
    y_true_window, y_pred_eval_best, average='binary', zero_division=0
)

cm_best = confusion_matrix(y_true_window, y_pred_eval_best)
tn_best, fp_best, fn_best, tp_best = cm_best.ravel()

roc_auc_best = roc_auc_score(y_true_window, eval_window_scores_best)
pr_auc_best = average_precision_score(y_true_window, eval_window_scores_best)
accuracy_best = accuracy_score(y_true_window, y_pred_eval_best)
balanced_acc_best = balanced_accuracy_score(y_true_window, y_pred_eval_best)
mcc_best = matthews_corrcoef(y_true_window, y_pred_eval_best)

metrics_best_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1",
        "MCC",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Value": [
        accuracy_best,
        balanced_acc_best,
        precision_best,
        recall_best,
        f1_best,
        mcc_best,
        roc_auc_best,
        pr_auc_best
    ]
})

print(metrics_best_df.to_string(index=False))
print("\nConfusion matrix:")
print(cm_best)

In [ ]:
# Visualisaties
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
fig.suptitle("Encoder-Only Transformer: Evaluatie van het beste model", fontsize=18, fontweight="bold")

# Plot 1: anomaly scores over de windows
ax1 = axes[0, 0]
ax1.plot(eval_window_scores_best, color="#1f77b4", linewidth=1.2, label="Anomaly score")
ax1.axhline(
    window_threshold_plot_best,
    color="#d62728",
    linestyle="--",
    linewidth=2,
    label=f"Plot threshold = {window_threshold_plot_best:.4f}"
)

anom_idx = np.where(y_true_window == 1)[0]
ax1.scatter(
    anom_idx,
    eval_window_scores_best[anom_idx],
    s=18,
    color="#ff7f0e",
    alpha=0.8,
    label="Ground truth anomalie",
    zorder=5
)

ax1.set_title("Anomaly scores per window")
ax1.set_xlabel("Window index")
ax1.set_ylabel("Score")
ax1.legend(loc="upper right")
ax1.grid(alpha=0.25)

# Plot 2: scoreverdeling per klasse
ax2 = axes[0, 1]
sns.histplot(
    eval_window_scores_best[y_true_window == 0],
    bins=40,
    kde=True,
    stat="density",
    color="#1f77b4",
    alpha=0.45,
    label="Normaal",
    ax=ax2
)
sns.histplot(
    eval_window_scores_best[y_true_window == 1],
    bins=40,
    kde=True,
    stat="density",
    color="#d62728",
    alpha=0.45,
    label="Anomalie",
    ax=ax2
)
ax2.axvline(window_threshold_plot_best, color="#2ca02c", linestyle="--", linewidth=2, label="Plot threshold")
ax2.set_title("Scoreverdeling per klasse")
ax2.set_xlabel("Score")
ax2.set_ylabel("Dichtheid")
ax2.legend(loc="upper right")
ax2.grid(alpha=0.25)

# Plot 3: confusion matrix
ax3 = axes[1, 0]
sns.heatmap(
    cm_best,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    square=True,
    xticklabels=["Voorspeld normaal", "Voorspeld anomalie"],
    yticklabels=["Werkelijk normaal", "Werkelijk anomalie"],
    ax=ax3
)
ax3.set_title("Confusion matrix")
ax3.set_xlabel("")
ax3.set_ylabel("")

# Plot 4: metric overview
ax4 = axes[1, 1]
metric_plot_df = metrics_best_df.sort_values("Value", ascending=True)

sns.barplot(
    data=metric_plot_df,
    x="Value",
    y="Metric",
    palette="viridis",
    ax=ax4
)

for i, value in enumerate(metric_plot_df["Value"]):
    ax4.text(min(value + 0.01, 1.02), i, f"{value:.3f}", va="center", fontsize=10)

ax4.set_xlim(0, max(1.0, metric_plot_df["Value"].max() + 0.15))
ax4.set_title("Overzicht van de kernmetrieken")
ax4.set_xlabel("Score")
ax4.set_ylabel("")

plt.show()

### 5.4 Beste model opslaan

In [ ]:
# Model opslaan
gebouw_naam = "dunant1"
huidige_datum = datetime.now().strftime("%Y-%m-%d")
model_dir = "models"
os.makedirs(model_dir, exist_ok=True) 

save_path = f"{model_dir}/encoder-only-{gebouw_naam}-{huidige_datum}.weights.h5"
best_model.save_weights(save_path)

print(f"Beste model succesvol opgeslagen in: {save_path}")

## STAP 6 - Inferentie: Het model in de praktijk gebruiken

In deze stap simuleren we hoe het systeem in de praktijk wordt gebruikt. We gaan ervan uit dat het model op een server draait en nieuwe, onvoorziene gebouwdata binnenkrijgt.

We doen het volgende:

1. We initialiseren het model en laden de getrainde gewichten in.
2. We voeden de (normale) testdata aan het model en berekenen de reconstructiefout.
3. We gebruiken de **Peak Over Threshold (POT)** methode (op basis van de Generalized Pareto Distribution) om de "baseline" drempelwaarde voor dit specifieke gebouw te berekenen.

### 6.1 Initialisatie model en laden gewichten

In [ ]:
# 1. Model opnieuw initialiseren en gewichten inladen (simulatie van praktijkgebruik)
# We gebruiken de best_params die we in Stap 5 hebben gevonden

best_params = best_hps.values

loaded_model = HVACAnomalyTransformer(
    num_features=NUM_FEATURES,
    d_model=best_params["d_model"],
    num_heads=NUM_HEADS,
    num_layers=best_params["num_layers"],
    ffn_expansion=best_params["ffn_expansion"],
    mask_ratio=best_params["mask_ratio"],
    block_len=best_params["block_len"],
    mask_seed=SEED
)

# Build for loading weights
_ = loaded_model(tf.zeros((1, WINDOW_SIZE, NUM_FEATURES), dtype=tf.float32))
loaded_model.load_weights(save_path)

# Alleen nodig als je opnieuw wil trainen/evalueren via compile/fit/evaluate
loaded_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"])
 )

print("Modelgewichten succesvol ingeladen met correcte hyperparameters.")

### 6.2 Voorspelling op ongeziene testdata

In [ ]:
# Voorspellingen doen op de ongeziene testdata
recon_test, attn_test = predict_with_attention(loaded_model, X_test, batch_size=BATCH_SIZE)
test_window_scores, test_feature_scores, _ = compute_attention_weighted_scores(
    X_test, recon_test, attn_test
 )

In [ ]:
print("Test window percentiles:", np.percentile(test_window_scores, [50, 90, 95, 99]))

### 6.3 POT-methode

De functie `fit_pot_thresholds` maakt gebruik van de **Extreme Waardetheorie (Extreme Value Theory - EVT)** om automatisch drempels te bepalen op basis van de aandacht-gewogen reconstructiefouten per feature.

Het algoritme doorloopt vier stappen:

**1. Initiële drempel ($t$) instellen**  
We starten met een baselinedrempel (standaard 85e percentiel) om enkel de top 15% hoogste waarden te behouden.

**2. Excessen bepalen ($Y_i$)**  
Voor de waarden boven $t$ berekenen we de excessen: $Y_i = X_i - t$.

**3. Generalized Pareto Distribution (GPD) fitten**  
De excessen worden gefit met een GPD (shape $\gamma$, scale $\sigma$).

**4. Dynamische drempel ($Z_q$) berekenen**  
Via de inverse CDF bepalen we de drempel voor risiconiveau $q$.

> Elke feature krijgt zijn eigen drempel. Een window wordt anomalie als **minstens één feature** de drempel overschrijdt.

### Praktische POT-uitleg (zoals in deze notebook)

We gebruiken **Peak Over Threshold (POT)** om automatisch drempels te bepalen per feature, zonder labels. In de paper wordt dit **rolling** gedaan met een venster van **2 uur** (hier: `ROLLING_WINDOW_STEPS`).

> Kort samengevat: POT kijkt alleen naar de **extreme staart** van de fouten en past een **Generalized Pareto Distribution (GPD)** om daar een drempel uit af te leiden.

**Stap 1: Anomaliescore berekenen**
- We nemen de absolute reconstructiefout per feature en wegen die met de attention-gewichten.
- Dit geeft een **timestep score per feature** (na het samenvoegen van overlap uit sliding windows).

**Stap 2: Rolling 2u venster**
- Voor elk tijdstip nemen we de laatste 2 uur aan scores (12 stappen bij 10-min data).
- Binnen dat venster nemen we een initiële drempel $t$ als het 85e percentiel (`POT_PERCENTILE`).

**Stap 3: Excessen en GPD-fit**
- Excessen: $Y_i = X_i - t$ voor alle $X_i > t$.
- We fitten een GPD op die excessen (met `genpareto.fit`, `loc=0`).

**Stap 4: Dynamische drempel**
- Met het gekozen risiconiveau $q$ (`POT_Q`) berekenen we $Z_q$ via de inverse CDF.
- Elke feature krijgt zo een **tijd-afhankelijke drempel**.

**Beslissing**
- Een timestep is anomalie als **minstens een feature** boven zijn drempel zit.
- Een window is anomalie als er **minstens een anomalous timestep** in zit.

In [ ]:
# Rolling POT thresholds op basis van ongeziene testdata (2u)
TEST_STRIDE = 1
weighted_test, _ = compute_weighted_errors(X_test, recon_test, attn_test)
timestep_scores_test = aggregate_to_timestep_series(
    weighted_test, len(test_norm), WINDOW_SIZE, TEST_STRIDE
 )
rolling_thresholds_test = rolling_pot_thresholds(
    timestep_scores_test, q=POT_Q, initial_percentile=POT_PERCENTILE, window_len=ROLLING_WINDOW_STEPS
 )
timestep_anom_test = (timestep_scores_test > rolling_thresholds_test).any(axis=1)
test_pred = window_labels_from_timestep(timestep_anom_test, WINDOW_SIZE, TEST_STRIDE)
print("Aandeel anomalous windows (test):", np.mean(test_pred))

In [ ]:
# Visualisatie: rolling POT drempels per feature (top 3)
feature_names = list(train_data.columns)
top_k = min(3, len(feature_names))
feature_order = np.argsort(timestep_scores_test.mean(axis=0))[::-1]
plot_features = feature_order[:top_k]
time_idx = np.arange(timestep_scores_test.shape[0])

fig, axes = plt.subplots(top_k, 1, figsize=(16, 4 * top_k), sharex=True)
if top_k == 1:
    axes = [axes]

for ax, f in zip(axes, plot_features):
    scores_f = timestep_scores_test[:, f]
    thresh_f = rolling_thresholds_test[:, f]
    anomalies_f = scores_f > thresh_f
    ax.plot(time_idx, scores_f, color="#1f77b4", linewidth=1.0, label="Score")
    ax.plot(time_idx, thresh_f, color="#d62728", linewidth=1.0, label="Rolling drempel")
    ax.scatter(time_idx[anomalies_f], scores_f[anomalies_f], s=8, color="#ff7f0e", label="Anomalie")
    ax.set_title(f"Feature: {feature_names[f]}")
    ax.set_ylabel("Score")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Timestep (10-min)  |  Rolling venster = 2u")
plt.show()

In [ ]:
# Deze stap gebeurt nu via rolling POT (2u) in de vorige cel.

## STAP 7 - Evaluatie op Gelabelde Synthetische Data

In deze stap evalueren we het best getrainde model op de synthetische testset met ground-truth labels (`0=normaal`, `1=anomalie`).

We doen vier dingen:

1. Synthetische data en labels inladen
2. De data met dezelfde scaler transformeren en omzetten naar sliding windows
3. Anomaliescores berekenen en window-level labels maken
4. Evalueren met de POT-threshold uit Stap 6 en met een extra getunede threshold op een tuning-split

In [ ]:
def score_windows_with_attention(model, X, batch_size=32):
    recon, attn = predict_with_attention(model, X, batch_size=batch_size)
    window_scores, feature_scores, _ = compute_attention_weighted_scores(X, recon, attn)
    return recon, window_scores, feature_scores

In [ ]:
synth_csv = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test.csv'
labels_npy = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test_labels.npy'

synth_df = pd.read_csv(synth_csv)
y_true_timestep = np.load(labels_npy).astype(int)

if 'timestamp' in synth_df.columns:
    synth_df = synth_df.drop(columns=['timestamp'])

for col in list(synth_df.columns):
    if col.lower().startswith('unnamed'):
        synth_df = synth_df.drop(columns=[col])

synth_df = synth_df[train_data.columns]
synth_norm = scaler.transform(synth_df)

STRIDE = 4
X_synth = create_windows(synth_norm, WINDOW_SIZE, step=STRIDE)

y_true_window = np.array([
    1 if np.max(y_true_timestep[i:i + WINDOW_SIZE]) > 0 else 0
    for i in range(0, len(y_true_timestep) - WINDOW_SIZE + 1, STRIDE)
], dtype=int)

# 7.2 Scores berekenen (window + timestep)
recon_synth_best, attn_synth_best = predict_with_attention(best_model, X_synth, batch_size=BATCH_SIZE)
scores_synth_window, scores_synth_feature, _ = compute_attention_weighted_scores(
    X_synth, recon_synth_best, attn_synth_best
 )

weighted_synth, _ = compute_weighted_errors(X_synth, recon_synth_best, attn_synth_best)
timestep_scores_synth = aggregate_to_timestep_series(
    weighted_synth, len(synth_norm), WINDOW_SIZE, STRIDE
 )
rolling_thresholds_synth = rolling_pot_thresholds(
    timestep_scores_synth, q=POT_Q, initial_percentile=POT_PERCENTILE, window_len=ROLLING_WINDOW_STEPS
 )
timestep_anom_synth = (timestep_scores_synth > rolling_thresholds_synth).any(axis=1)
y_pred_eval_best = window_labels_from_timestep(timestep_anom_synth, WINDOW_SIZE, STRIDE)

print("val window:", np.percentile(val_window_scores_best, [50, 90, 95, 99]))
print("eval window:", np.percentile(scores_synth_window, [1, 5, 50, 95, 99]))
print("plot threshold (p99 val):", window_threshold_plot_best)
print("eval below plot threshold:", np.mean(scores_synth_window <= window_threshold_plot_best))
print("shapes:", "y_true_window =", y_true_window.shape, "| y_pred_eval_best =", y_pred_eval_best.shape)

# 7.4 Scorekaart exact zoals 5.3
precision_best, recall_best, f1_best, _ = precision_recall_fscore_support(
    y_true_window, y_pred_eval_best, average='binary', zero_division=0
)

cm_best = confusion_matrix(y_true_window, y_pred_eval_best)
tn_best, fp_best, fn_best, tp_best = cm_best.ravel()

roc_auc_best = roc_auc_score(y_true_window, scores_synth_window)
pr_auc_best = average_precision_score(y_true_window, scores_synth_window)
accuracy_best = accuracy_score(y_true_window, y_pred_eval_best)
balanced_acc_best = balanced_accuracy_score(y_true_window, y_pred_eval_best)
mcc_best = matthews_corrcoef(y_true_window, y_pred_eval_best)

metrics_best_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1",
        "MCC",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Value": [
        accuracy_best,
        balanced_acc_best,
        precision_best,
        recall_best,
        f1_best,
        mcc_best,
        roc_auc_best,
        pr_auc_best
    ]
})

print(metrics_best_df.to_string(index=False))
print("\nConfusion matrix:")
print(cm_best)

In [ ]:
# Visualisaties
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
fig.suptitle("Encoder-Only Transformer: Evaluatie van het beste model", fontsize=18, fontweight="bold")

# Plot 1: anomaly scores over de windows
ax1 = axes[0, 0]
ax1.plot(scores_synth_window, color="#1f77b4", linewidth=1.2, label="Anomaly score")
ax1.axhline(
    window_threshold_plot_best,
    color="#d62728",
    linestyle="--",
    linewidth=2,
    label=f"Plot threshold = {window_threshold_plot_best:.4f}"
)

anom_idx = np.where(y_true_window == 1)[0]
ax1.scatter(
    anom_idx,
    scores_synth_window[anom_idx],
    s=18,
    color="#ff7f0e",
    alpha=0.8,
    label="Ground truth anomalie",
    zorder=5
)

ax1.set_title("Anomaly scores per window")
ax1.set_xlabel("Window index")
ax1.set_ylabel("Score")
ax1.legend(loc="upper right")
ax1.grid(alpha=0.25)

# Plot 2: scoreverdeling per klasse
ax2 = axes[0, 1]
sns.histplot(
    scores_synth_window[y_true_window == 0],
    bins=40,
    kde=True,
    stat="density",
    color="#1f77b4",
    alpha=0.45,
    label="Normaal",
    ax=ax2
)
sns.histplot(
    scores_synth_window[y_true_window == 1],
    bins=40,
    kde=True,
    stat="density",
    color="#d62728",
    alpha=0.45,
    label="Anomalie",
    ax=ax2
)
ax2.axvline(window_threshold_plot_best, color="#2ca02c", linestyle="--", linewidth=2, label="Plot threshold")
ax2.set_title("Scoreverdeling per klasse")
ax2.set_xlabel("Score")
ax2.set_ylabel("Dichtheid")
ax2.legend(loc="upper right")
ax2.grid(alpha=0.25)

# Plot 3: confusion matrix
ax3 = axes[1, 0]
sns.heatmap(
    cm_best,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    square=True,
    xticklabels=["Voorspeld normaal", "Voorspeld anomalie"],
    yticklabels=["Werkelijk normaal", "Werkelijk anomalie"],
    ax=ax3
)
ax3.set_title("Confusion matrix")
ax3.set_xlabel("")
ax3.set_ylabel("")

# Plot 4: metric overview
ax4 = axes[1, 1]
metric_plot_df = metrics_best_df.sort_values("Value", ascending=True)

sns.barplot(
    data=metric_plot_df,
    x="Value",
    y="Metric",
    palette="viridis",
    ax=ax4
)

for i, value in enumerate(metric_plot_df["Value"]):
    ax4.text(min(value + 0.01, 1.02), i, f"{value:.3f}", va="center", fontsize=10)

ax4.set_xlim(0, max(1.0, metric_plot_df["Value"].max() + 0.15))
ax4.set_title("Overzicht van de kernmetrieken")
ax4.set_xlabel("Score")
ax4.set_ylabel("")

plt.show()